In [2]:
import numpy as np
import polars as pl
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import GammaRegressor
from sklearn.base import clone
from sklearn.metrics import mean_gamma_deviance, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression

In [3]:
train = pl.read_csv("train_predict_cost.csv")
# Surfaces: surface_farmland, surface_grassland, surface_road, surface_structure, surface_swamp, surface_water
# Materials: material_brass, material_cast_iron, material_copper, material_gray_iron, material_wrought_iron
train.describe()

statistic,pipe_id,date,time,cost,lay_date,material,surface,gps_x1,gps_x2,gps_y1,gps_y2,pipe_length,log1p_pipe_length,years_used_at_leak,log1p_years_used,midpoint_x,midpoint_y,event_decimal_year,event_year,event_month,season_sin,season_cos,material_code,surface_code,material_brass,material_cast_iron,material_copper,material_gray_iron,material_wrought_iron,surface_farmland,surface_grassland,surface_road,surface_structure,surface_swamp,surface_water
str,str,str,str,f64,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""14107""","""14107""","""14107""",14107.0,"""14107""","""14107""","""14107""",14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0,14107.0
"""null_count""","""0""","""0""","""0""",0.0,"""0""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",null,null,null,25021.64337,null,null,null,3385.419417,3385.365536,3516.085821,3515.408745,19.408258,2.718278,63.326879,4.160271,3385.392476,3515.747283,2023.803756,2023.254058,7.134118,-0.155028,-0.016992,2.824768,2.678812,0.069327,0.229673,0.065003,0.078897,0.557099,0.147161,0.270504,0.05919,0.134189,0.057418,0.331538
"""std""",null,null,null,156118.062761,null,null,null,1431.786412,1431.74212,1414.118358,1414.151009,16.962142,0.757649,5.251382,0.088904,1431.740175,1414.100385,2.151476,2.153006,3.376555,0.672644,0.723393,1.460885,1.935543,0.254019,0.420638,0.24654,0.269587,0.496747,0.354279,0.444236,0.235989,0.340867,0.232648,0.470782
"""min""","""P01002""","""2019-01-06""","""00:00""",19.78,"""1945-01-14""","""brass""","""farmland""",350.44,342.81,306.14,307.53,0.17,0.157004,34.2423,3.562247,346.625,306.835,2019.013699,2019.0,1.0,-0.999991,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",null,null,null,1426.37,null,null,null,2046.42,2046.57,2361.74,2363.5,8.38767,2.239397,62.483231,4.150776,2046.675,2362.585,2022.367123,2022.0,5.0,-0.790946,-0.762354,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""50%""",null,null,null,3277.14,null,null,null,3658.4,3658.53,3586.03,3587.92,11.613957,2.534804,63.471595,4.166225,3658.52,3587.745,2023.775342,2023.0,8.0,-0.288482,-0.03012,4.0,3.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
"""75%""",null,null,null,9247.62,null,null,null,4390.16,4387.65,4511.65,4513.02,25.670002,3.283539,64.372348,4.180099,4389.445,4513.085,2025.79726,2025.0,10.0,0.46355,0.714673,4.0,5.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
"""max""","""P44036""","""2026-12-31""","""23:59""",7.2576e6,"""1985-11-27""","""wrought iron""","""water""",7349.37,7349.38,7149.85,7149.85,74.555194,4.324863,81.771389,4.416082,7349.375,7149.85,2026.99726,2026.0,12.0,0.999991,1.0,4.0,5.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [4]:
bin_cols = ["surface_farmland", "surface_road", "surface_structure",
            "surface_swamp", "surface_water",
            "material_brass", "material_copper",
            "material_gray_iron", "material_wrought_iron"]
num_cols = ["log1p_pipe_length", "log1p_years_used", "season_sin", "season_cos", "event_decimal_year"]

df = train.drop_nulls(subset=["cost"] + bin_cols + num_cols)

feature_cols = num_cols + bin_cols
X = df.select(feature_cols).to_numpy().astype(float)
y = df["cost"].to_numpy()

n_num = len(num_cols)
pre = ColumnTransformer(
    [("num", StandardScaler(), list(range(n_num)))],   
    remainder="passthrough",                           
)

In [6]:
### Gamma Regression ###

years = df["event_year"].to_numpy()

model = make_pipeline(pre, GammaRegressor(alpha=0, max_iter=1000))

rows = []
for test_year in [2023, 2024, 2025, 2026]:
    tr = years < test_year          
    te = years == test_year

    m = clone(model).fit(X[tr], y[tr])      
    p = m.predict(X[te])

    rows.append({
        "train": f"2019-{test_year-1}",
        "test": test_year,
        "n_train": tr.sum(),
        "n_test": te.sum(),
        "rmse": np.sqrt(np.mean((y[te] - p) ** 2)),
        "mae": mean_absolute_error(y[te], p),
        "gamma_dev": mean_gamma_deviance(y[te], p),
        "mean_pred": p.mean(),
        "mean_actual": y[te].mean(),
    })

for r in rows:
    print(f"train {r['train']} -> test {r['test']}: "
          f"n={r['n_train']}/{r['n_test']} | RMSE {r['rmse']:,.0f} | MAE {r['mae']:,.0f} | "
          f"gamma dev {r['gamma_dev']:.4f} | mean pred/actual {r['mean_pred']:,.0f}/{r['mean_actual']:,.0f}")

print("\nmean RMSE across the 4 splits:", f"{np.mean([r['rmse'] for r in rows]):,.0f}",
      "| sd:", f"{np.std([r['rmse'] for r in rows], ddof=1):,.0f}")

print("\nmean MAE across the 4 splits:", f"{np.mean([r['mae'] for r in rows]):,.0f}",
      "| sd:", f"{np.std([r['mae'] for r in rows], ddof=1):,.0f}")

train 2019-2022 -> test 2023: n=5521/1824 | RMSE 160,506 | MAE 29,911 | gamma dev 1.8137 | mean pred/actual 24,564/23,405
train 2019-2023 -> test 2024: n=7345/1951 | RMSE 102,551 | MAE 15,741 | gamma dev 1.6818 | mean pred/actual 11,798/14,380
train 2019-2024 -> test 2025: n=9296/1773 | RMSE 114,922 | MAE 24,915 | gamma dev 1.6627 | mean pred/actual 22,691/24,059
train 2019-2025 -> test 2026: n=11069/3038 | RMSE 129,501 | MAE 24,132 | gamma dev 1.7470 | mean pred/actual 19,580/23,387

mean RMSE across the 4 splits: 126,870 | sd: 24,983

mean MAE across the 4 splits: 23,675 | sd: 5,876


In [8]:
np.mean([r['rmse'] for r in rows])

np.float64(126870.17720484847)

In [42]:
### Linear Regression ###

model = make_pipeline(pre, LinearRegression())

rows = []
for test_year in [2023, 2024, 2025, 2026]:
    tr = years < test_year
    te = years == test_year

    m = clone(model).fit(X[tr], y[tr])
    p = m.predict(X[te])

    rows.append({
        "train": f"2019-{test_year-1}",
        "test": test_year,
        "n_train": tr.sum(),
        "n_test": te.sum(),
        "rmse": np.sqrt(np.mean((y[te] - p) ** 2)),
        "mae": mean_absolute_error(y[te], p),
        "n_nonpos_pred": int((p <= 0).sum()),
        "mean_pred": p.mean(),
        "mean_actual": y[te].mean(),
    })

for r in rows:
    print(f"train {r['train']} -> test {r['test']}: "
          f"n={r['n_train']}/{r['n_test']} | RMSE {r['rmse']:,.0f} | MAE {r['mae']:,.0f} | "
          f"non-positive preds {r['n_nonpos_pred']} | mean pred/actual {r['mean_pred']:,.0f}/{r['mean_actual']:,.0f}")
    
    print("\nmean RMSE across the 4 splits:", f"{np.mean([r['rmse'] for r in rows]):,.0f}",
      "| sd:", f"{np.std([r['rmse'] for r in rows], ddof=1):,.0f}")

print("\nmean MAE across the 4 splits:", f"{np.mean([r['mae'] for r in rows]):,.0f}",
      "| sd:", f"{np.std([r['mae'] for r in rows], ddof=1):,.0f}")

train 2019-2022 -> test 2023: n=5521/1824 | RMSE 159,753 | MAE 36,824 | non-positive preds 301 | mean pred/actual 29,282/23,405

mean RMSE across the 4 splits: 125,851 | sd: 25,478
train 2019-2023 -> test 2024: n=7345/1951 | RMSE 100,001 | MAE 17,849 | non-positive preds 652 | mean pred/actual 9,639/14,380

mean RMSE across the 4 splits: 125,851 | sd: 25,478
train 2019-2024 -> test 2025: n=9296/1773 | RMSE 114,846 | MAE 28,081 | non-positive preds 503 | mean pred/actual 24,227/24,059

mean RMSE across the 4 splits: 125,851 | sd: 25,478
train 2019-2025 -> test 2026: n=11069/3038 | RMSE 128,805 | MAE 27,737 | non-positive preds 1293 | mean pred/actual 17,834/23,387

mean RMSE across the 4 splits: 125,851 | sd: 25,478

mean MAE across the 4 splits: 27,623 | sd: 7,755


In [1]:
### Random Forest ###

bin_cols = ["surface_farmland", "surface_road", "surface_structure",
            "surface_swamp", "surface_water", "surface_grassland",
            "material_brass", "material_copper", "material_cast_iron",
            "material_gray_iron", "material_wrought_iron"]
num_cols = ["log1p_pipe_length", "log1p_years_used", "season_sin", "season_cos"]

df = train.drop_nulls(subset=["cost"] + bin_cols + num_cols)

feature_cols = num_cols + bin_cols
X = df.select(feature_cols).to_numpy().astype(float)
y = df["cost"].to_numpy()

n_num = len(num_cols)
pre = ColumnTransformer(
    [("num", StandardScaler(), list(range(n_num)))],   
    remainder="passthrough",                           
)

# Regular version
# model = make_pipeline(
#     pre,
#     RandomForestRegressor(n_estimators=500, min_samples_leaf=5, n_jobs=-1, random_state=0),
# )

# Log version
model = TransformedTargetRegressor(
    regressor=make_pipeline(
        pre,
        RandomForestRegressor(n_estimators=500, min_samples_leaf=5, n_jobs=-1, random_state=0),
    ),
    func=np.log,
    inverse_func=np.exp,
)



rows = []
for test_year in [2023, 2024, 2025, 2026]:
    tr = years < test_year
    te = years == test_year

    m = clone(model).fit(X[tr], y[tr])
    p = m.predict(X[te])

    rows.append({
        "train": f"2019-{test_year-1}",
        "test": test_year,
        "n_train": tr.sum(),
        "n_test": te.sum(),
        "rmse": np.sqrt(np.mean((y[te] - p) ** 2)),
        "mae": mean_absolute_error(y[te], p),
        "mean_pred": p.mean(),
        "mean_actual": y[te].mean(),
    })
    
for r in rows:
    print(f"train {r['train']} -> test {r['test']}: "
          f"n={r['n_train']}/{r['n_test']} | RMSE {r['rmse']:,.0f} | MAE {r['mae']:,.0f} | "
          f"mean pred/actual {r['mean_pred']:,.0f}/{r['mean_actual']:,.0f}")

print("\nmean RMSE across the 4 splits:", f"{np.mean([r['rmse'] for r in rows]):,.0f}",
      "| sd:", f"{np.std([r['rmse'] for r in rows], ddof=1):,.0f}")

print("\nmean MAE across the 4 splits:", f"{np.mean([r['mae'] for r in rows]):,.0f}",
      "| sd:", f"{np.std([r['mae'] for r in rows], ddof=1):,.0f}")

NameError: name 'train' is not defined